# CSV-Export: Mapillary Coverage für OSM Highways

Dieses Skript kombiniert die Coverage-Daten (Pano + Regular) aus Schritt 3 und erstellt:
- Eine finale CSV-Datei mit allen OSM-Highway-Segmenten, die >= 60% Mapillary-Coverage haben
- Eine README.md mit Zusammenfassung nach Bundesland

In [1]:
import pandas as pd
import geopandas as gpd

from datetime import datetime
from pathlib import Path
import json
import glob
import os

from config import TILES_CONFIG, PROCESSING_CONFIG, MAPILLARY_CONFIG, GEOFABRIK_CONFIG

# Konfiguration
OUTPUT_FOLDER = PROCESSING_CONFIG["output_folder"]
COVERAGE_THRESHOLD = PROCESSING_CONFIG['mp_coverage_ratio_threshold']

In [2]:
def should_process_csv(output_file, max_age_days=None):
    """
    Prüft ob CSV neu erstellt werden muss.
    
    Args:
        output_file: Pfad zur CSV-Datei
        max_age_days: Maximales Alter in Tagen (default: aus config)
    
    Returns:
        True wenn CSV-Erstellung nötig, False sonst
    """
    if max_age_days is None:
        max_age_days = PROCESSING_CONFIG.get('max_file_age_days', 4)
    
    if not os.path.exists(output_file):
        return True
    
    try:
        file_age_days = (datetime.now().timestamp() - os.path.getmtime(output_file)) / 86400
        
        if file_age_days >= max_age_days:
            print(f"  ℹ️  CSV ist {file_age_days:.1f} Tage alt → Erneuerung nötig")
            return True
        else:
            print(f"  ✓ CSV ist {file_age_days:.1f} Tage alt → aktuell genug")
            return False
            
    except Exception as e:
        print(f"  ⚠️  Fehler beim Prüfen: {e}")
        return True

In [3]:
# Prüfe ob CSV-Export nötig ist
csv_output_file = f"{OUTPUT_FOLDER}/germany_osm-highways_mp-coverage_latest.csv"

print(f"{'='*70}")
print(f"CSV-Export Prüfung")
print(f"{'='*70}")

if not should_process_csv(csv_output_file):
    print(f"\n✅ CSV ist aktuell genug - keine Verarbeitung nötig!")
    print(f"   Datei: {csv_output_file}")
    print(f"   Wenn du trotzdem neu erstellen möchtest, lösche die Datei.")
    SHOULD_PROCESS = False
else:
    print(f"\n▶️  Fahre fort mit CSV-Erstellung...")
    SHOULD_PROCESS = True

print(f"{'='*70}")

CSV-Export Prüfung
  ℹ️  CSV ist 7.0 Tage alt → Erneuerung nötig

▶️  Fahre fort mit CSV-Erstellung...


In [4]:
#SHOULD_PROCESS = True


In [5]:
def load_metadata():
    """Lädt OSM und Mapillary Metadaten aus JSON-Dateien."""
    files = {
        "output/osm_metadata.json": "osm_data_from",
        "output/ml_metadata.json": "ml_data_from",
    }
    
    metadata = {
        "osm_data_from": None, 
        "ml_data_from": None,
        "osm_bundeslaender": {},
        "ml_bundeslaender": {}
    }
    
    for fname, primary_key in files.items():
        p = Path(fname)
        if not p.exists():
            print(f"⚠️  {p} nicht gefunden")
            continue
        
        with p.open("r", encoding="utf-8") as f:
            meta = json.load(f)
        
        if primary_key in meta:
            metadata[primary_key] = meta[primary_key]
        
        # Lade auch Bundesland-spezifische Timestamps
        if "bundeslaender" in meta:
            if "osm" in fname:
                metadata["osm_bundeslaender"] = meta["bundeslaender"]
            elif "ml" in fname:
                metadata["ml_bundeslaender"] = meta["bundeslaender"]
    
    return metadata

# Lade Metadaten
metadata = load_metadata()
print(f"OSM Daten von: {metadata['osm_data_from']}")
print(f"Mapillary Daten von: {metadata['ml_data_from']}")
print(f"OSM Bundesländer: {len(metadata['osm_bundeslaender'])}")
print(f"ML Bundesländer: {len(metadata['ml_bundeslaender'])}")

OSM Daten von: 2026-02-07T21:21:12Z
Mapillary Daten von: 2026-02-16T01:28:25.385777+00:00
OSM Bundesländer: 16
ML Bundesländer: 16


In [6]:
def load_coverage_data(coverage_type="pano"):
    """
    Lädt Coverage-Daten für alle Bundesländer.
    
    Args:
        coverage_type: "pano" oder "regular"
    
    Returns:
        tuple: (combined_df, summary_list)
    """
    # Finde Dateien
    files = glob.glob(f"{OUTPUT_FOLDER}/DE-*_osm-highways_mp_{coverage_type}_coverage_latest.parquet")
    
    # Filter nach ausgewählten Bundesländern
    selected_bundeslaender = GEOFABRIK_CONFIG.get("bundeslaender")
    if selected_bundeslaender:
        files = [f for f in files if any(bl in f for bl in selected_bundeslaender)]
    
    print(f"\n{'─'*70}")
    print(f"Lade {coverage_type.upper()} Coverage-Daten:")
    print(f"  Dateien gefunden: {len(files)}")
    
    all_dfs = []
    summary_list = []
    
    for file in files:
        # Lade nur benötigte Spalten
        df = pd.read_parquet(file, columns=["osm_id", "mp_coverage_ratio", "length_m_before_clip"])
        
        # Extrahiere Bundesland
        bundesland = file.split('/')[-1].split('_')[0]
        
        # Summary-Statistik
        total_length = df['length_m_before_clip'].sum()
        summary_list.append({
            'Bundesland': bundesland,
            'Typ': coverage_type,
            'Anzahl Segmente': len(df),
            'Gesamtlänge (km)': total_length / 1000
        })
        
        all_dfs.append(df)
        print(f"    {bundesland}: {len(df):,} Segmente, {total_length/1000:,.2f} km")
    
    # Kombiniere alle Bundesländer
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        print(f"  ✓ Gesamt: {len(combined_df):,} Segmente")
        return combined_df, summary_list
    else:
        print(f"  ⚠️  Keine Daten gefunden!")
        return pd.DataFrame(columns=["osm_id", "mp_coverage_ratio"]), []

# Lade Daten nur wenn nötig
if SHOULD_PROCESS:
    gdf_pano, pano_summary = load_coverage_data("pano")
    gdf_regular, regular_summary = load_coverage_data("regular")
else:
    gdf_pano, pano_summary = pd.DataFrame(), []
    gdf_regular, regular_summary = pd.DataFrame(), []


──────────────────────────────────────────────────────────────────────
Lade PANO Coverage-Daten:
  Dateien gefunden: 15
    DE-BW: 126,312 Segmente, 18,954.51 km
    DE-TH: 6,662 Segmente, 1,627.56 km
    DE-BB: 129,382 Segmente, 26,061.48 km
    DE-MV: 7,260 Segmente, 2,829.28 km
    DE-NW: 25,321 Segmente, 3,035.59 km
    DE-SH: 7,893 Segmente, 1,654.33 km


    DE-RP: 29,356 Segmente, 7,854.45 km
    DE-BY: 94,299 Segmente, 18,774.84 km


    DE-ST: 46,586 Segmente, 8,784.99 km
    DE-SN: 36,936 Segmente, 3,844.47 km


    DE-HH: 3,604 Segmente, 323.33 km
    DE-NI: 43,296 Segmente, 11,633.12 km
    DE-BE: 196,767 Segmente, 13,407.38 km
    DE-HB: 855 Segmente, 114.27 km
    DE-HE: 46,837 Segmente, 7,855.75 km
  ✓ Gesamt: 801,366 Segmente

──────────────────────────────────────────────────────────────────────
Lade REGULAR Coverage-Daten:
  Dateien gefunden: 16


    DE-NI: 325,356 Segmente, 70,482.61 km
    DE-HH: 81,363 Segmente, 6,806.35 km
    DE-SH: 98,836 Segmente, 22,157.71 km


    DE-HE: 211,267 Segmente, 34,345.53 km


    DE-MV: 43,469 Segmente, 10,806.42 km
    DE-NW: 602,157 Segmente, 92,397.23 km
    DE-BE: 185,365 Segmente, 12,387.13 km
    DE-TH: 71,703 Segmente, 14,267.48 km
    DE-SL: 29,000 Segmente, 5,180.66 km


    DE-BW: 322,638 Segmente, 51,506.92 km
    DE-HB: 10,630 Segmente, 1,315.77 km


    DE-BY: 427,670 Segmente, 78,867.27 km
    DE-SN: 186,843 Segmente, 27,776.79 km
    DE-BB: 177,903 Segmente, 45,356.77 km
    DE-RP: 204,312 Segmente, 40,868.10 km
    DE-ST: 93,468 Segmente, 20,115.06 km


  ✓ Gesamt: 3,071,980 Segmente


## Zusammenfassung nach Bundesland

In [7]:
if SHOULD_PROCESS and (pano_summary or regular_summary):
    # Kombiniere Summaries
    summary_df = pd.concat([
        pd.DataFrame(pano_summary),
        pd.DataFrame(regular_summary)
    ], ignore_index=True).sort_values(['Bundesland', 'Typ'])
    
    print(f"\n{'='*70}")
    print("📊 Zusammenfassung nach Bundesland und Typ:")
    print(f"{'='*70}")
    print(summary_df.to_string(index=False))
    
    # Gesamtsummen
    total_pano_km = summary_df[summary_df['Typ']=='pano']['Gesamtlänge (km)'].sum()
    total_regular_km = summary_df[summary_df['Typ']=='regular']['Gesamtlänge (km)'].sum()
    total_pano_count = summary_df[summary_df['Typ']=='pano']['Anzahl Segmente'].sum()
    total_regular_count = summary_df[summary_df['Typ']=='regular']['Anzahl Segmente'].sum()
    
    print(f"\n{'='*70}")
    print(f"Gesamt Pano:    {total_pano_km:,.2f} km ({total_pano_count:,} Segmente)")
    print(f"Gesamt Regular: {total_regular_km:,.2f} km ({total_regular_count:,} Segmente)")
    print(f"{'='*70}")
    
    display(summary_df)
else:
    summary_df = pd.DataFrame()
    print("⏩ Keine Summary zu erstellen")


📊 Zusammenfassung nach Bundesland und Typ:
Bundesland     Typ  Anzahl Segmente  Gesamtlänge (km)
     DE-BB    pano           129382      26061.477001
     DE-BB regular           177903      45356.770601
     DE-BE    pano           196767      13407.380863
     DE-BE regular           185365      12387.128958
     DE-BW    pano           126312      18954.513454
     DE-BW regular           322638      51506.920881
     DE-BY    pano            94299      18774.837658
     DE-BY regular           427670      78867.270378
     DE-HB    pano              855        114.268870
     DE-HB regular            10630       1315.772939
     DE-HE    pano            46837       7855.751420
     DE-HE regular           211267      34345.532864
     DE-HH    pano             3604        323.329028
     DE-HH regular            81363       6806.345863
     DE-MV    pano             7260       2829.281195
     DE-MV regular            43469      10806.415250
     DE-NI    pano            43296   

,Bundesland,Typ,Anzahl Segmente,Gesamtlänge (km)
2,DE-BB,pano,129382,26061.477001
28,DE-BB,regular,177903,45356.770601
12,DE-BE,pano,196767,13407.380863
21,DE-BE,regular,185365,12387.128958
0,DE-BW,pano,126312,18954.513454
24,DE-BW,regular,322638,51506.920881
7,DE-BY,pano,94299,18774.837658
26,DE-BY,regular,427670,78867.270378
13,DE-HB,pano,855,114.268870
25,DE-HB,regular,10630,1315.772939


## Filtern nach Coverage-Threshold und Kombinieren

In [8]:
def filter_and_tag(df, coverage_type):
    """
    Filtert DataFrame nach Coverage-Threshold und fügt Tag hinzu.
    
    Args:
        df: DataFrame mit osm_id und mp_coverage_ratio
        coverage_type: "pano" oder "regular"
    
    Returns:
        DataFrame mit osm_id und mapillary_coverage
    """
    if df.empty:
        return pd.DataFrame(columns=["osm_id", "mapillary_coverage"])
    
    # Behalte nur benötigte Spalten
    filtered = df[["osm_id", "mp_coverage_ratio"]].copy()
    
    # Filter nach Threshold
    filtered = filtered[filtered["mp_coverage_ratio"] >= COVERAGE_THRESHOLD].copy()
    
    # Füge Coverage-Type hinzu
    filtered["mapillary_coverage"] = coverage_type
    
    # Entferne mp_coverage_ratio
    filtered = filtered[["osm_id", "mapillary_coverage"]]
    
    print(f"  {coverage_type}: {len(filtered):,} Segmente über Threshold ({COVERAGE_THRESHOLD*100:.0f}%)")
    
    return filtered

if SHOULD_PROCESS:
    print(f"\n{'─'*70}")
    print(f"Filtere nach Coverage-Threshold >= {COVERAGE_THRESHOLD*100:.0f}%:")
    
    pano_filtered = filter_and_tag(gdf_pano, "pano")
    regular_filtered = filter_and_tag(gdf_regular, "regular")
else:
    pano_filtered = pd.DataFrame()
    regular_filtered = pd.DataFrame()


──────────────────────────────────────────────────────────────────────
Filtere nach Coverage-Threshold >= 60%:
  pano: 450,486 Segmente über Threshold (60%)


  regular: 1,696,914 Segmente über Threshold (60%)


In [9]:
if SHOULD_PROCESS:
    # Kombiniere Pano + Regular
    both_concat = pd.concat([pano_filtered, regular_filtered], ignore_index=True)
    
    print(f"\n{'─'*70}")
    print(f"Vor Duplikat-Entfernung:")
    print(both_concat["mapillary_coverage"].value_counts().to_string())
    
    # Entferne Duplikate: Pano hat Vorrang über Regular
    # (ascending=True bei sort → 'pano' kommt vor 'regular' alphabetisch)
    both_concat = both_concat.sort_values(
        by="mapillary_coverage", 
        ascending=True
    ).drop_duplicates(
        subset="osm_id", 
        keep="first"
    )
    
    print(f"\nNach Duplikat-Entfernung (Pano bevorzugt):")
    print(both_concat["mapillary_coverage"].value_counts().to_string())
    print(f"\nGesamt eindeutige Segmente: {len(both_concat):,}")
    
    display(both_concat.head(10))
else:
    both_concat = pd.DataFrame()
    print("⏩ Keine Kombination nötig")


──────────────────────────────────────────────────────────────────────
Vor Duplikat-Entfernung:


mapillary_coverage
regular    1696914
pano        450486



Nach Duplikat-Entfernung (Pano bevorzugt):
mapillary_coverage
regular    1497129
pano        450425

Gesamt eindeutige Segmente: 1,947,554


,osm_id,mapillary_coverage
0,1450732492,pano
300330,35139661,pano
300329,35133590,pano
300328,35104560,pano
300327,35104559,pano
300326,35097117,pano
300325,35097037,pano
300324,35096723,pano
300323,35079637,pano
300322,35072916,pano


In [10]:
## CSV und README exportieren

In [11]:
if SHOULD_PROCESS and not both_concat.empty:
    # Speichere CSV
    both_concat.to_csv(csv_output_file, index=False)
    
    print(f"\n{'='*70}")
    print(f"✅ CSV exportiert:")
    print(f"   {csv_output_file}")
    print(f"   {len(both_concat):,} Zeilen")
    print(f"{'='*70}")
else:
    print("\n⏩ Kein CSV-Export")


✅ CSV exportiert:
   output/germany_osm-highways_mp-coverage_latest.csv
   1,947,554 Zeilen


In [12]:
def create_readme(summary_df, metadata):
    """Erstellt README.md mit Zusammenfassung."""
    current_date = datetime.now().strftime("%Y-%m-%d")
    
    # Pivot-Tabelle für Markdown
    summary_table = summary_df.pivot(
        index='Bundesland', 
        columns='Typ', 
        values='Gesamtlänge (km)'
    ).fillna(0).reset_index()
    
    # Hole Bundesland-spezifische Timestamps
    osm_bl = metadata.get("osm_bundeslaender", {})
    ml_bl = metadata.get("ml_bundeslaender", {})
    
    # Markdown-Tabelle erstellen mit zusätzlichen Datums-Spalten
    table_header = "| Bundesland | Pano (km) | Regular (km) | OSM Datum | Mapillary Datum |\n|------------|-----------|--------------|-----------|-----------------|"
    table_rows = []
    
    for _, row in summary_table.iterrows():
        bundesland = row['Bundesland']
        pano_km = f"{row.get('pano', 0):,.2f}" if 'pano' in row else "0.00"
        regular_km = f"{row.get('regular', 0):,.2f}" if 'regular' in row else "0.00"
        
        # Extrahiere Datum (nur YYYY-MM-DD)
        osm_date = osm_bl.get(bundesland, "N/A")
        ml_date = ml_bl.get(bundesland, "N/A")
        
        if osm_date != "N/A":
            osm_date = osm_date.split('T')[0]
        if ml_date != "N/A":
            ml_date = ml_date.split('T')[0]
        
        table_rows.append(f"| {bundesland} | {pano_km} | {regular_km} | {osm_date} | {ml_date} |")
    
    markdown_table = "\n".join([table_header] + table_rows)
    
    # README-Inhalt
    osm_date = metadata['osm_data_from'].split('T')[0] if metadata['osm_data_from'] else "N/A"
    ml_date = metadata['ml_data_from'].split('T')[0] if metadata['ml_data_from'] else "N/A"
    
    readme_content = f"""# Mapillary Coverage per OSM Highway — Output

This folder contains the **latest** output file for *Mapillary coverage per OSM highway analysis*.

| Property | Value |
|----------|-------|
| **Data created** | {current_date} |
| **OSM data** | {osm_date} |
| **Mapillary data** | {PROCESSING_CONFIG['min_capture_date']} → {ml_date} |
| **Buffer distance** | {PROCESSING_CONFIG['buffer_distance']} meters |
| **Coverage ratio threshold** | {PROCESSING_CONFIG['mp_coverage_ratio_threshold']} ({int(PROCESSING_CONFIG['mp_coverage_ratio_threshold']*100)}%) |

Segments are considered *covered* if at least {int(PROCESSING_CONFIG['mp_coverage_ratio_threshold']*100)}% of their length falls within {PROCESSING_CONFIG['buffer_distance']} meters of Mapillary images.

## Summary by Bundesland

{markdown_table}
"""
    
    return readme_content

if SHOULD_PROCESS and not summary_df.empty:
    readme_content = create_readme(summary_df, metadata)
    
    readme_path = f"{OUTPUT_FOLDER}/README.md"
    with open(readme_path, "w", encoding="utf-8") as f:
        f.write(readme_content)
    
    print(f"\n✅ README erstellt: {readme_path}")
else:
    print("\n⏩ Kein README-Export")

print(f"\n{'='*70}")
print("✅ FERTIG!")
print(f"{'='*70}")


✅ README erstellt: output/README.md

✅ FERTIG!
